# Week 95, Day 4: Opening the pull request

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w095d4.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('fix.diff', 'w', encoding='utf-8') as f:
    f.write('diff --git a/Lib/wraptext.py b/Lib/wraptext.py\n--- a/Lib/wraptext.py\n+++ b/Lib/wraptext.py\n@@ -14,7 +14,8 @@ def shorten(text, width, placeholder=" [...]"):\n     kept, length = [], 0\n     for word in words:\n         extra = len(word) + (1 if kept else 0)\n-        if length + extra + len(placeholder) >= width:\n+        print(word, length + extra)\n+        if length + extra + len(placeholder) > width:\n             break\n         kept.append(word)\n         length += extra\ndiff --git a/Lib/test/test_wraptext.py b/Lib/test/test_wraptext.py\n--- a/Lib/test/test_wraptext.py\n+++ b/Lib/test/test_wraptext.py\n@@ -20,3 +20,8 @@ class TestShorten(unittest.TestCase):\n     def test_fits_already(self):\n         self.assertEqual(shorten("Hello world", 40), "Hello world")\n \n+    def test_exact_fit_is_kept(self):\n+        # gh-77: text that exactly fills the width fits \n+        self.assertEqual(shorten("Hello a nice day today", 18), "Hello a nice [...]")\n+        # TODO: more widths\n+\ndiff --git a/Misc/NEWS.d/next/Library/2026-11-20-14-03-12.gh-issue-77.AbCdEf.rst b/Misc/NEWS.d/next/Library/2026-11-20-14-03-12.gh-issue-77.AbCdEf.rst\nnew file mode 100644\n--- /dev/null\n+++ b/Misc/NEWS.d/next/Library/2026-11-20-14-03-12.gh-issue-77.AbCdEf.rst\n@@ -0,0 +1,2 @@\n+Fix :func:`wraptext.shorten` dropping a word that exactly fills the width.\n+Patch by Ada Walker.\ndiff --git a/.vscode/settings.json b/.vscode/settings.json\nnew file mode 100644\n--- /dev/null\n+++ b/.vscode/settings.json\n@@ -0,0 +1,3 @@\n+{\n+    "editor.rulers": [79]\n+}\n')

**Predict before you run:** what will this print?



In [ ]:
diff = open("fix.diff").read()
print([line.split(" b/", 1)[1] for line in diff.splitlines() if line.startswith("diff --git")])

> **Key idea.** Commit messages: an imperative subject without a full stop (aim for 50 characters, at most 72), a blank line, and a body wrapped at 72 saying why. Read your diff hunk by hunk; every added line should be needed. Remove debug prints, `breakpoint()`, `TODO`s, trailing whitespace and unrelated files. Describe the problem, the change, and how you tested it, linking the issue. Before opening: up to date, tests pass, `make patchcheck`, NEWS entry, title convention.

### Exercise: Checking a commit message

Write `commit_problems(message)`: ignore trailing newlines; if the message is empty, return `["empty message"]`. Otherwise report, in order: `"subject over 72 characters"`; `"subject ends with a full stop"`; `"subject should use the imperative (WORD)"` if the subject's first word (after any `gh-N: ` prefix) ends in `ed` or `ing`, or is one of `Fixes`, `Adds`, `Removes`, `Updates`, `Changes`, `Makes`, `Uses`; `"no blank line after the subject"` if there's a second line that isn't empty; and `"body line N over 72 characters"` for each longer line from the third (N counting from 1).

In [ ]:
import re

def commit_problems(message):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("GOOD = \"\"\"gh-77: Keep a word that exactly fills the width\n\nshorten() stopped one word early when the kept words and the\nplaceholder exactly filled the width, because the loop compared with\n>= instead of >. Restore the comparison and add a regression test.\n\"\"\"\n\ndef test_good():\n    \"a message worth reading\"\n    assert commit_problems(GOOD) == []\n\ndef test_bad():\n    \"everything wrong\"\n    assert commit_problems(\"Fixed the shorten bug.\\nIt was wrong.\\n\" + \"x\" * 80) == [\n        \"subject ends with a full stop\", \"subject should use the imperative (Fixed)\", \"no blank line after the subject\",\n        \"body line 3 over 72 characters\"]\n    assert commit_problems(\"gh-77: Fixes shorten\") == [\"subject should use the imperative (Fixes)\"]\n    assert commit_problems(\"\\n\\n\") == [\"empty message\"] and commit_problems(\"A\" * 73) == [\"subject over 72 characters\"]")

### Exercise: Wrapping the body

Write `wrap_body(text, width=72)`: split the text into paragraphs at blank lines; leave a paragraph unchanged if every line of it starts with four spaces (code); otherwise join its lines with spaces and wrap it with `textwrap.fill`. Return the paragraphs joined by a blank line.

In [ ]:
import textwrap

def wrap_body(text, width=72):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_wrap():\n    \"72 columns, code untouched\"\n    text = (\"shorten() stopped one word early when the kept words and the placeholder exactly filled the width, because the loop compared with >= instead of >.\\n\\n\"\n            \"    >>> shorten('Hello a nice day today', 18)\\n    'Hello a nice [...]'\\n\\nRestore the comparison.\")\n    out = wrap_body(text)\n    paragraphs = out.split(\"\\n\\n\")\n    assert len(paragraphs) == 3 and all(len(line) <= 72 for line in paragraphs[0].splitlines())\n    assert paragraphs[0].splitlines()[0] == \"shorten() stopped one word early when the kept words and the placeholder\"\n    assert paragraphs[1] == \"    >>> shorten('Hello a nice day today', 18)\\n    'Hello a nice [...]'\" and paragraphs[2] == \"Restore the comparison.\"\n    assert wrap_body(\"one\\ntwo\", width=72) == \"one two\"")

### Exercise: What's in the diff?

Write `diff_files(diff)`: return `(PATH, ADDED, REMOVED)` for each file, in order, taking the path from the part of its `diff --git` line after ` b/`, and counting lines starting with `+` and `-` but not the `+++` and `---` lines.

In [ ]:
def diff_files(diff):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_files():\n    \"four files, one unexpected\"\n    assert diff_files(open(\"fix.diff\").read()) == [\n        (\"Lib/wraptext.py\", 2, 1), (\"Lib/test/test_wraptext.py\", 5, 0),\n        (\"Misc/NEWS.d/next/Library/2026-11-20-14-03-12.gh-issue-77.AbCdEf.rst\", 2, 0), (\".vscode/settings.json\", 3, 0)]\n    assert diff_files(\"\") == []")

### Exercise: Reviewing your own diff

Write a generator `added_lines(diff)` yielding `(PATH, LINE, TEXT)` for each added line, with LINE its number in the new file (start from the `+` number in each hunk header; added and unchanged lines move it on, removed lines don't). Then `self_review(diff, allowed)` reporting, for each added line, `"PATH:LINE: debug print"` (a `print(` in a file under `Lib/` but not `Lib/test/`), `"PATH:LINE: debugger left in"` (`breakpoint()` or `import pdb`), `"PATH:LINE: TODO"` and `"PATH:LINE: trailing whitespace"`, then `"unrelated file: PATH"` for each changed file that doesn't start with one of the `allowed` prefixes.

In [ ]:
import re

HUNK = re.compile(r"@@ -\d+(?:,\d+)? \+(\d+)(?:,\d+)? @@")

def added_lines(diff):
    ___

def self_review(diff, allowed):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_review():\n    \"what a reviewer would find\"\n    diff = open(\"fix.diff\").read()\n    assert list(added_lines(diff))[:2] == [(\"Lib/wraptext.py\", 17, \"        print(word, length + extra)\"),\n                                           (\"Lib/wraptext.py\", 18, \"        if length + extra + len(placeholder) > width:\")]\n    assert self_review(diff, [\"Lib/\", \"Misc/NEWS.d/\"]) == [\n        \"Lib/wraptext.py:17: debug print\", \"Lib/test/test_wraptext.py:24: trailing whitespace\",\n        \"Lib/test/test_wraptext.py:26: TODO\", \"unrelated file: .vscode/settings.json\"]\n    tiny = \"diff --git a/Lib/x.py b/Lib/x.py\\n--- a/Lib/x.py\\n+++ b/Lib/x.py\\n@@ -3,2 +3,2 @@\\n-old\\n+breakpoint()\\n same\\n\"\n    assert self_review(tiny, [\"Lib/\"]) == [\"Lib/x.py:3: debugger left in\"]")

### Exercise: The description

Write `pr_body(issue, summary, changes, testing)`, returning Markdown lines joined by newlines: the summary, a blank line, `Changes:`, a `- ` line per change, a blank line, `Testing:`, a `- ` line per item, a blank line, and `* Issue: gh-ISSUE`.

In [ ]:
def pr_body(issue, summary, changes, testing):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_body():\n    \"for a reviewer arriving cold\"\n    text = pr_body(77, \"shorten() dropped a word that exactly filled the width.\",\n                   [\"Compare with > instead of >= in the loop\", \"Add a regression test and boundary tests\"],\n                   [\"./python -m test test_wraptext\", \"make patchcheck\"])\n    assert text == (\"shorten() dropped a word that exactly filled the width.\\n\\nChanges:\\n- Compare with > instead of >= in the loop\\n\"\n                    \"- Add a regression test and boundary tests\\n\\nTesting:\\n- ./python -m test test_wraptext\\n- make patchcheck\\n\\n* Issue: gh-77\")")

### Bug hunt: A full stop that slipped through

The message ends with a newline, so `endswith(".")` on the whole message misses the full stop at the end of the subject, and this prints `[]`. Check the subject line, so it prints `['subject ends with a full stop']`.

In [ ]:
message = "Keep a word that exactly fills the width.\n"
problems = []
if message.endswith("."):
    problems.append("subject ends with a full stop")
print(problems)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"the first line\"\n    expect_output(\"['subject ends with a full stop']\")")

### Bug hunt: A header counted as a change

The `+++` line names the new file; it isn't an added line. Counting it gives `2` here. Skip the header so it prints `1`.

In [ ]:
diff = "--- a/Lib/x.py\n+++ b/Lib/x.py\n@@ -1 +1 @@\n-old\n+new\n"
print(sum(1 for line in diff.splitlines() if line.startswith("+")))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"headers aren't changes\"\n    expect_output(\"1\")")

### Puzzle: Numbering the added lines

Arrange the loop that numbers added lines in the new file: hunk headers set the number; added and unchanged lines move it on. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the mood a commit subject is written in (one word)
q = ___   # the most characters a commit body line should have
r = ___   # the new hunk's starting line in "@@ -14,7 +14,8 @@"
s = ___   # True or False: a .vscode settings file belongs in a CPython fix
t = ___   # the make target that runs CPython's own pre-review checks

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == \"imperative\" and q == 72 and r == 14, \"p, q and r\"\n    assert s is False and t == \"patchcheck\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: ready to open?

Write `pr_ready(message, title, diff, issue, allowed)` with the given helpers. Return `(lines, verdict)`:

1. `"commit: ok"` or `"commit: "` and its problems joined by `"; "`;
2. `"title: ok"` or `"title: "` and its problems joined by `"; "`;
3. `"test: ok"` if a changed file is under `Lib/test/`, else `"test: none changed"`;
4. `"news: ok"` if a changed file is under `Misc/NEWS.d/` and contains `gh-issue-ISSUE.`, else `"news: missing"`;
5. `"self-review: ok"`, or `"self-review: N problems"` followed by each problem indented by two spaces.

`verdict` is `"open it"` if every check is ok, otherwise `"fix first: "` and the failing check names joined by `", "`.

In [ ]:
import re

NOT_IMPERATIVE = {"Fixes", "Adds", "Removes", "Updates", "Changes", "Makes", "Uses"}

def commit_problems(message):
    lines = message.rstrip("\n").splitlines()
    if not lines or not lines[0].strip():
        return ["empty message"]
    subject = lines[0]
    problems = []
    if len(subject) > 72:
        problems.append("subject over 72 characters")
    if subject.endswith("."):
        problems.append("subject ends with a full stop")
    summary = re.sub(r"^gh-\d+: ", "", subject)
    first = summary.split()[0] if summary.split() else ""
    if first.endswith(("ed", "ing")) or first in NOT_IMPERATIVE:
        problems.append(f"subject should use the imperative ({first})")
    if len(lines) > 1 and lines[1] != "":
        problems.append("no blank line after the subject")
    problems += [f"body line {n} over 72 characters" for n, line in enumerate(lines[2:], start=3) if len(line) > 72]
    return problems

def title_problems(title, issue):
    prefix = f"gh-{issue}: "
    problems = []
    summary = title[len(prefix):] if title.startswith(prefix) else title
    if not title.startswith(prefix):
        problems.append(f"should start with {prefix}")
    if not summary[:1].isupper():
        problems.append("summary should start with a capital letter")
    if title.endswith("."):
        problems.append("no full stop at the end")
    if len(title) > 72:
        problems.append("over 72 characters")
    return problems

def diff_files(diff):
    files = []
    for line in diff.splitlines():
        if line.startswith("diff --git "):
            files.append([line.split(" b/", 1)[1], 0, 0])
        elif line.startswith(("+++", "---")):
            continue
        elif files and line.startswith("+"):
            files[-1][1] += 1
        elif files and line.startswith("-"):
            files[-1][2] += 1
    return [tuple(f) for f in files]

HUNK = re.compile(r"@@ -\d+(?:,\d+)? \+(\d+)(?:,\d+)? @@")

def diff_files(diff):
    files = []
    for line in diff.splitlines():
        if line.startswith("diff --git "):
            files.append([line.split(" b/", 1)[1], 0, 0])
        elif line.startswith(("+++", "---")):
            continue
        elif files and line.startswith("+"):
            files[-1][1] += 1
        elif files and line.startswith("-"):
            files[-1][2] += 1
    return [tuple(f) for f in files]

def added_lines(diff):
    path, number = None, 0
    for line in diff.splitlines():
        if line.startswith("diff --git "):
            path = line.split(" b/", 1)[1]
        elif line.startswith(("+++", "---")):
            continue
        elif m := HUNK.match(line):
            number = int(m.group(1))
        elif line.startswith("+"):
            yield path, number, line[1:]
            number += 1
        elif line.startswith(" "):
            number += 1

def self_review(diff, allowed):
    problems = []
    for path, n, text in added_lines(diff):
        where = f"{path}:{n}"
        if path.startswith("Lib/") and not path.startswith("Lib/test/") and re.search(r"\bprint\(", text):
            problems.append(f"{where}: debug print")
        if "breakpoint()" in text or "import pdb" in text:
            problems.append(f"{where}: debugger left in")
        if "TODO" in text:
            problems.append(f"{where}: TODO")
        if text != text.rstrip():
            problems.append(f"{where}: trailing whitespace")
    problems += [f"unrelated file: {p}" for p, _, _ in diff_files(diff) if not any(p.startswith(a) for a in allowed)]
    return problems

def pr_ready(message, title, diff, issue, allowed):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("MESSAGE = \"\"\"gh-77: Keep a word that exactly fills the width\n\nshorten() stopped one word early when the kept words and the\nplaceholder exactly filled the width. Restore the comparison.\n\"\"\"\nALLOWED = [\"Lib/\", \"Misc/NEWS.d/\"]\n\ndef test_first_draft():\n    \"the diff as it was\"\n    lines, verdict = pr_ready(MESSAGE, \"gh-77: Keep a word that exactly fills the width\", open(\"fix.diff\").read(), 77, ALLOWED)\n    assert lines == [\"commit: ok\", \"title: ok\", \"test: ok\", \"news: ok\", \"self-review: 4 problems\",\n                     \"  Lib/wraptext.py:17: debug print\", \"  Lib/test/test_wraptext.py:24: trailing whitespace\",\n                     \"  Lib/test/test_wraptext.py:26: TODO\", \"  unrelated file: .vscode/settings.json\"]\n    assert verdict == \"fix first: self-review\"\n\ndef test_cleaned():\n    \"after cleaning up\"\n    diff = open(\"fix.diff\").read()\n    diff = diff.split(\"diff --git a/.vscode\")[0]\n    diff = diff.replace(\"+        print(word, length + extra)\\n\", \"\").replace(\"@@ -14,7 +14,8 @@\", \"@@ -14,7 +14,7 @@\")\n    diff = diff.replace(\"fits \\n\", \"fits\\n\").replace(\"+        # TODO: more widths\\n\", \"\").replace(\"+20,8\", \"+20,7\")\n    lines, verdict = pr_ready(MESSAGE, \"gh-77: Keep a word that exactly fills the width\", diff, 77, ALLOWED)\n    assert lines[-1] == \"self-review: ok\" and verdict == \"open it\"\n    lines, verdict = pr_ready(\"Fixed it.\", \"keep words\", diff.split(\"diff --git a/Lib/test\")[0], 78, ALLOWED)\n    assert verdict == \"fix first: commit, title, test, news\"")

## Warm-up

Four files, and one of them, `.vscode/settings.json`, has nothing to do with the fix: an editor saved its settings and `git add .` picked them up. `git diff --stat` or `git status` before committing would have shown it. Reviewers notice; so should you, first.

## Commit messages

A commit message has a **subject** line, a blank line, and a **body**. The subject says what the commit does, in the **imperative** ("Keep a word that exactly fills the width", as if completing "This commit will..."), without a full stop, short enough to read in a list (aim for 50 characters, never more than 72). The body, wrapped at 72 characters, says **why**: what was wrong, what caused it, and how the change fixes it. In CPython the pull request is squashed into one commit when merged, titled with the pull request's title (`gh-77: ...`), and the commits on your branch matter less; elsewhere, they're the project's history. Write them well everywhere.

## Reading your own diff

A **unified diff** (what `git diff` prints) has, per file, a `diff --git a/PATH b/PATH` line, `---` and `+++` lines naming the old and new versions, and **hunks**: a header such as `@@ -14,7 +14,8 @@` (the old hunk starts at line 14 and has 7 lines; the new one starts at 14 and has 8), then lines starting with a space (unchanged), `-` (removed) or `+` (added). Counting from the header tells you the line number of every added line in the new file.

Read every added line, asking: is this needed for the fix or its test? Look especially for debug output, `breakpoint()`, `TODO`s, trailing whitespace (CPython's CI checks it), commented-out code, and files that don't belong.

## The description

The description is for a reviewer arriving cold. Say what the problem was in a sentence, link the issue (`gh-77`; CPython's bots link it from the title too), say what you changed and why this approach, and say **how you tested it**, including the regression test. Keep it short; point at the issue for the full story. If you're unsure about something, say so: "I wasn't sure whether the docs should mention this; happy to add a sentence." Open a **draft** pull request if it isn't ready for review yet.

## Pressing the button

Before opening it: the branch is up to date with `main`, the tests pass on your build, `patchcheck` is happy (`make patchcheck` runs CPython's own checks for whitespace and NEWS entries), your diff contains only what it should, the NEWS entry is there, and the title follows the convention. Then open it, and wait: CI runs, the CLA bot checks, and reviewers come when they can.

## Practice

## Quiz

## Daily challenge

## Your turn

1. **Look at your diff** (`git diff main --stat`, then `git diff main`), and remove everything that isn't the fix, its test and its NEWS entry.
2. **Write the commit message** in the imperative with a wrapped body, and the NEWS entry with `blurb add`.
3. **Run `make patchcheck`** and the relevant tests on your build.
4. **Open the pull request** against `python/cpython`'s `main`, titled `gh-NNNNN: ...`, with a short description and how you tested it. Then wait patiently.

## Before you go

Tomorrow is about what happens next: review comments, suggestions, requests for changes, and how to answer them well.